### Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

### define paths

In [2]:
TRAIN_DIR = Path("../dataset/split/train")
VAL_DIR = Path("../dataset/split/val")
TEST_DIR = Path("../dataset/split/test")

MODEL_PATH = Path(
    "../backend/trained_models/mobilenetv2_transfer.keras"
)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

### Load datasets

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names

print(class_names)

Found 6238 files belonging to 8 classes.
Found 1337 files belonging to 8 classes.
Found 1337 files belonging to 8 classes.
['CCI_Caterpillars', 'CCI_Leaflets', 'Gray Leaf Spot', 'Healthy_Leaves', 'Leaf Rot', 'WCLWD_DryingofLeaflets', 'WCLWD_Flaccidity', 'WCLWD_Yellowing']


### Then prefetch

In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

### Recreate class weights

In [5]:
weight_lookup = {
    "CCI_Caterpillars": 1.125180,
    "CCI_Leaflets": 1.402428,
    "Gray Leaf Spot": 0.522621,
    "Healthy_Leaves": 9.066860,
    "Leaf Rot": 0.678634,
    "WCLWD_DryingofLeaflets": 1.032781,
    "WCLWD_Flaccidity": 1.042447,
    "WCLWD_Yellowing": 1.027339,
}

class_weights = {
    index: weight_lookup[class_name]
    for index, class_name in enumerate(class_names)
}

class_weights

{0: 1.12518,
 1: 1.402428,
 2: 0.522621,
 3: 9.06686,
 4: 0.678634,
 5: 1.032781,
 6: 1.042447,
 7: 1.027339}

### Load the saved transfer-learning model

In [6]:
mobilenet_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "preprocess_input": preprocess_input
    }
)

In [7]:
for index, layer in enumerate(mobilenet_model.layers):
    print(index, layer.name, type(layer).__name__)

0 data_augmentation Sequential
1 lambda Lambda
2 mobilenetv2_1.00_224 Functional
3 global_average_pooling2d_1 GlobalAveragePooling2D
4 dropout_1 Dropout
5 dense_1 Dense


### Get the MobileNetV2 base

In [8]:
mobilenet_base = mobilenet_model.layers[2]

print(mobilenet_base.name)
print("Total base layers:", len(mobilenet_base.layers))

mobilenetv2_1.00_224
Total base layers: 154


### Unfreeze only the last 30 layers

In [9]:
mobilenet_base.trainable = True

for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

In [12]:
# keep Batch Normalization layers frozen
for layer in mobilenet_base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

In [13]:
# Check how many layers are trainable
trainable_count = sum(
    layer.trainable
    for layer in mobilenet_base.layers
)

print("Trainable MobileNetV2 layers:", trainable_count)

Trainable MobileNetV2 layers: 19


### Recompile with a very small learning rate

In [14]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

### Fine-tuning callbacks

In [15]:
finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "../backend/trained_models/mobilenetv2_finetuned.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

### Fine-tune

In [16]:
FINETUNE_EPOCHS = 10

mobilenet_finetune_history = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=finetune_callbacks
)

Epoch 1/10
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 941ms/step - accuracy: 0.9684 - loss: 0.0870
Epoch 1: val_loss improved from None to 0.11352, saving model to ../backend/trained_models/mobilenetv2_finetuned.keras

Epoch 1: finished saving model to ../backend/trained_models/mobilenetv2_finetuned.keras
195/195 ━━━━━━━━━━━━━━━━━━━━ 221s 1s/step - accuracy: 0.9711 - loss: 0.0816 - val_accuracy: 0.9574 - val_loss: 0.1135 - learning_rate: 1.0000e-05
Epoch 2/10
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 924ms/step - accuracy: 0.9740 - loss: 0.0715
Epoch 2: val_loss improved from 0.11352 to 0.07126, saving model to ../backend/trained_models/mobilenetv2_finetuned.keras

Epoch 2: finished saving model to ../backend/trained_models/mobilenetv2_finetuned.keras
195/195 ━━━━━━━━━━━━━━━━━━━━ 257s 1s/step - accuracy: 0.9760 - loss: 0.0653 - val_accuracy: 0.9768 - val_loss: 0.0713 - learning_rate: 1.0000e-05
Epoch 3/10
195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 928ms/step - accuracy: 0.9751 - loss: 0.0539
Epoch 3: val_loss improve

### Test and predict

In [17]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

mobilenet_finetuned = tf.keras.models.load_model(
    "../backend/trained_models/mobilenetv2_finetuned.keras",
    custom_objects={
        "preprocess_input": preprocess_input
    }
)

In [18]:
y_true_ft = []
y_pred_ft = []

for images, labels in test_ds:
    predictions = mobilenet_finetuned.predict(images, verbose=0)

    y_true_ft.extend(
        np.argmax(labels.numpy(), axis=1)
    )

    y_pred_ft.extend(
        np.argmax(predictions, axis=1)
    )

y_true_ft = np.array(y_true_ft)
y_pred_ft = np.array(y_pred_ft)

In [19]:
from sklearn.metrics import accuracy_score, classification_report

ft_test_accuracy = accuracy_score(
    y_true_ft,
    y_pred_ft
)

print(
    "Fine-Tuned MobileNetV2 Test Accuracy:",
    round(ft_test_accuracy, 4)
)

print(
    classification_report(
        y_true_ft,
        y_pred_ft,
        target_names=class_names,
        digits=4
    )
)

Fine-Tuned MobileNetV2 Test Accuracy: 0.9918
                        precision    recall  f1-score   support

      CCI_Caterpillars     1.0000    1.0000    1.0000       148
          CCI_Leaflets     1.0000    1.0000    1.0000       119
        Gray Leaf Spot     1.0000    0.9906    0.9953       320
        Healthy_Leaves     0.8571    0.9474    0.9000        19
              Leaf Rot     0.9960    0.9960    0.9960       247
WCLWD_DryingofLeaflets     0.9817    0.9938    0.9877       162
      WCLWD_Flaccidity     0.9937    0.9812    0.9874       160
       WCLWD_Yellowing     0.9816    0.9877    0.9846       162

              accuracy                         0.9918      1337
             macro avg     0.9763    0.9871    0.9814      1337
          weighted avg     0.9920    0.9918    0.9918      1337

